In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
from torch_mlir import fx

repo_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(repo_root))
from tutorial._infra import torch_frontend as torch_nb

class GemmModule(nn.Module):
    def forward(self, inputs):
        A, B = inputs
        return A @ B

m = GemmModule().eval()

A = torch.randn(1024, 2048, dtype=torch.float32)
B = torch.randn(2048, 512, dtype=torch.float32)
example_input = (A, B)

tm = fx.export_and_import(m, example_input, func_name="kernel")
torch_ir = tm.operation.get_asm()

out_file = torch_nb.ARTIFACTS_DIR / "gemm_torch.mlir"
out_file.write_text(torch_ir)
print(f"Wrote Torch dialect IR → {out_file.resolve()}")
print(torch_ir)


In [ ]:
from tutorial._infra import torch_frontend as torch_nb

pipeline = (
    "builtin.module("
    "torch-function-to-torch-backend-pipeline,"
    "torch-backend-to-linalg-on-tensors-backend-pipeline,"
    "torch-verify-linalg-on-tensors-backend-contract"
    ")"
)

torch_ir = torch_nb.ARTIFACTS_DIR / "gemm_torch.mlir"
linalg_ir = torch_nb.ARTIFACTS_DIR / "gemm_linalg.mlir"

torch_nb.run(
    [
        torch_nb.torch_mlir_opt,
        torch_ir,
        f"-pass-pipeline={pipeline}",
        "-o",
        linalg_ir,
    ]
)

print(f"Wrote Linalg IR → {linalg_ir}")
print(linalg_ir.read_text())

In [ ]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "func.func(cinm-cleanup-linalg,im2col-to-matmul)"
    ")"
)

gemm_linalg_im2col_clean = torch_nb.ARTIFACTS_DIR / "gemm_linalg_im2col_clean.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        linalg_ir,  
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_linalg_im2col_clean,
    ]
)

print(gemm_linalg_im2col_clean.read_text())

In [ ]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "func.func(convert-linalg-to-cinm)"
    ")"
)

gemm_cinm0 = torch_nb.ARTIFACTS_DIR / "gemm_cinm0.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_linalg_im2col_clean, 
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm0,
    ]
)

print(gemm_cinm0.read_text())

In [ ]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "cinm-annotate-tiles{ops=gemv tile-sizes=32x32},"
    "cinm-annotate-tiles{ops=gemv tile-sizes=32x32},"
    "cinm-annotate-tiles{ops=gemm tile-sizes=32x32x32},"
    "cinm-annotate-tiles{ops=batch_gemv tile-sizes=32x32x32},"
    "cinm-annotate-tiles{ops=batch_gemm tile-sizes=32x32x32x32},"
    "cinm-annotate-tiles{ops=activate tile-sizes=32}"
    ")"
)

gemm_cinm1 = torch_nb.ARTIFACTS_DIR / "gemm_cinm1.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm0,  
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm1,
    ]
)

print(gemm_cinm1.read_text())

In [ ]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "cinm-gemm-to-gemv{split-dim=2}"
    ")"
)

gemm_cinm2 = torch_nb.ARTIFACTS_DIR / "gemm_cinm2.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm1,  
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm2,
    ]
)

print(gemm_cinm2.read_text())

In [ ]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "cinm-tiling"
    ")"
)

gemm_cinm3 = torch_nb.ARTIFACTS_DIR / "gemm_cinm3.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm2, 
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm3,
    ]
)

print(gemm_cinm3.read_text())

In [ ]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "lower-affine,scf-for-loop-canonicalization,"
    "func.func(cinm-gemv-min-write)"
    ")"
)

gemm_cinm4 = torch_nb.ARTIFACTS_DIR / "gemm_cinm4.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm3, 
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm4,
    ]
)

print(gemm_cinm4.read_text())

In [ ]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "func.func(cinm-decompose-accum)"
    ")"
)

gemm_cinm5 = torch_nb.ARTIFACTS_DIR / "gemm_cinm5.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm4, 
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm5,
    ]
)

print(gemm_cinm5.read_text())

In [ ]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "func.func(cinm-insert-quantization{ops=gemm,gemv qtype=i8 scale=0.03125 zp=0 rounding=nearest narrow-range=false})"
    ")"
)

gemm_cinm6 = torch_nb.ARTIFACTS_DIR / "gemm_cinm6.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm5, 
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm6,
    ]
)

print(gemm_cinm5.read_text())


In [ ]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "lower-affine,"
    "func.func(cinm-relower)"
    ")"
)

gemm_cinm7 = torch_nb.ARTIFACTS_DIR / "gemm_cinm7.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm6, 
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm7,
    ]
)

print(gemm_cinm7.read_text())

In [ ]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "func.func(linalg-generalize-named-ops,canonicalize,scf-for-loop-canonicalization),"
    "one-shot-bufferize{bufferize-function-boundaries}"
    ")"
)

gemm_cinm7 = torch_nb.ARTIFACTS_DIR / "gemm_cinm7.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm6,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm7,
    ]
)

print(gemm_cinm7.read_text())


In [ ]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "cinm-memory-cleanup,"
    "func.func(convert-cinm-to-cim,cim-mark-relower{ops=add,relu})"
    ")"
)

gemm_cinm8 = torch_nb.ARTIFACTS_DIR / "gemm_cinm8.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm7, 
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm8,
    ]
)

print(gemm_cinm8.read_text())

In [ ]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "func.func(convert-cim-to-alpine,cim-cleanup-unsupported),"
    "func.func(alpine-hoist-write-weights)"
    ")"
)

gemm_cinm9 = torch_nb.ARTIFACTS_DIR / "gemm_cinm9.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm8, 
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm9,
    ]
)

print(gemm_cinm9.read_text())

In [ ]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "convert-alpine-to-func,"
    "convert-linalg-to-loops,lower-affine,convert-scf-to-cf,canonicalize,"
    "resolve-ranked-shaped-type-result-dims,expand-strided-metadata,"
    "memref-expand,canonicalize,lower-affine,canonicalize,"
    "convert-vector-to-llvm,convert-math-to-llvm,convert-arith-to-llvm,"
    "convert-index-to-llvm,convert-scf-to-cf,convert-cf-to-llvm,"
    "convert-func-to-llvm,finalize-memref-to-llvm,canonicalize,"
    "convert-to-llvm,reconcile-unrealized-casts,"
    "canonicalize"
    ")"
)

gemm_cinm9 = torch_nb.ARTIFACTS_DIR / "gemm_cinm9.mlir"
gemm_cinm10 = torch_nb.ARTIFACTS_DIR / "gemm_cinm10.mlir"

cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm9,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm10,
    ]
)

print(gemm_cinm10.read_text())